In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import kagglehub

# 1. Load and inspect data

In [ ]:
df = pd.read_csv('/kaggle/input/datasets/vedavyasv/usa-housing/USA_Housing.csv')
df.head()

In [ ]:
df.info()

In [ ]:
print(df.shape)
print(df.isnull().sum())
print(df.duplicated().sum())
print(df.describe().T)       

# 2. Univariate — Histograms + KDE

In [ ]:
numeric_cols = ['Avg. Area Income','Avg. Area House Age','Avg. Area Number of Rooms',
               'Avg. Area Number of Bedrooms','Area Population', 'Price'
               ]

fig, axes = plt.subplots(2,3, figsize=(16,9))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    sns.histplot(df[col], kde=True, ax=axes[i], color='#2196F3', bins=40)
    axes[i].set_title(col)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count (density)')

plt.tight_layout()
plt.savefig('univariate_histograms.png', dpi=100)


In [ ]:
print(df[numeric_cols].skew())

# 3. Univariate — Boxplots (Outlier Detection)

In [ ]:
fig, axes = plt.subplots(2,3, figsize=(16,8))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    axes[i].boxplot(df[col], vert=True, patch_artist=True,
                   boxprops=dict(facecolor='#90CAF9'))
    axes[i].set_title(col)

plt.tight_layout()
plt.savefig('boxplots.png', dpi=100)

# IQR method
for col in numeric_cols:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = df[(df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)]
    print(f"{col}: {len(outliers)} outliers ({len(outliers)/len(df)*100:.1f}%)")

# 4. Bivariate — Scatter Plots (Har Feature vs Price)

In [ ]:
features = ['Avg. Area Income', 'Avg. Area House Age', 'Avg. Area Number of Rooms',
            'Avg. Area Number of Bedrooms', 'Area Population']

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(features):
    axes[i].scatter(df[col], df['Price'], alpha=0.15, s=10, color='#1976D2')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Price')
    corr = df[col].corr(df['Price'])
    axes[i].set_title(f'{col}\ncorr={corr:.3f}')

axes[5].axis('off')
plt.tight_layout()
plt.savefig('bivariate_scatter.png', dpi=100)

# 5. Correlation Heatmap

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(df[numeric_cols].corr(), annot=True, fmt='.2f',
           cmap='coolwarm', center=0, square=True)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=100)

# 6. Feature Engineering

In [ ]:
df['State'] = df['Address'].str.extract(r',\s*([A-Z]{2})\s')

print(df['State'].nunique()) # 59 - USA has only 50 states
print(df['State'].isnull().sum())

df = df.drop(columns=['Address']) # drop Address column